# BeTiSe — Period Detection on 200 Series: STL vs MSTL + Seasonal ARMA Evidence

**Goal:** Evaluate period detection on **80 seasonal** + **120 nonseasonal** BeTiSe Requested Generation series, including nonseasonal integrated ARI/IMA/ARIMA controls. No generation code or GitHub repository files are modified by preparing this notebook.

| Set | Series types | Number | True seasonal periods |
|---|---|---:|---|
| Seasonal | single_seasonality, multiple_seasonality, sarma, sarima | 20 each (80) | From metadata (evaluation only) |
| Nonseasonal controls | ar, ma, arma, ari, ima, arima | 20 each (120) | **Empty set `[]`** |

The algorithms receive **only numeric time series**, with a global candidate grid `[7, 12, 24, 52]`. The ground truth is used only for calculating accuracy.

Four outputs are compared, without changing the original voting decisions:

- `stl`: original detector using single-period STL seasonal strengths.
- `mstl_top2`: original voting, but top-two candidate strengths are evaluated jointly with MSTL.
- `stochastic_proposal_only`: a **separate** ACF + seasonal ARMA-model comparison that recommends **zero or one** period. It does not use STL/MSTL decisions.
- `fallback_only`: an **experimental rule**: use the MSTL prediction when it is nonempty, otherwise use the stochastic proposal. Not a final/general detector.

**Important:** An empty prediction `[]` is the **correct** result for all 120 nonseasonal controls. A nonempty prediction on those controls is a false alarm. High autocorrelation in integrated ARI/IMA/ARIMA can easily be mistaken for seasonal structure; the experiment tests exactly that.

This notebook **does not fit any seasonal differencing/integration candidate** in the stochastic comparison. Seasonal AR/MA model alternatives all have `D=0`. No seasonal unit-root test or stochastic/deterministic type classifier is implemented. Seasonal component recovery is out of scope.


## 1. Configuration and dependencies

Run the cells from your BeTiSe repository root. Edit `DATA_ROOT` if you generated your data elsewhere. `statsmodels >= 0.14` is required for `MSTL`; `pyarrow` or `fastparquet` is needed to read Parquet. Your repository's `requirements.txt` includes `pyarrow`, but currently only requires `statsmodels>=0.13`, so check your environment.

Change your four Requested Generation JSON files to `"num_series": 20`. Existing `"shard_size": 10` is fine: that creates two Parquet shards for each category.

In [1]:
from pathlib import Path
import ast
import json
import warnings

import numpy as np
import pandas as pd
from scipy.signal import detrend as scipy_detrend
from statsmodels.tsa.seasonal import STL
import statsmodels

try:
    from statsmodels.tsa.seasonal import MSTL
except ImportError as exc:
    raise ImportError("MSTL requires statsmodels >= 0.14; upgrade your environment.") from exc

from IPython.display import display

print("statsmodels:", statsmodels.__version__)

DATA_ROOT = Path("generated-dataset/seasonality_pilot")
OUTPUT_ROOT = Path.cwd().parent / "BeTiSe_period_detection_200series_results"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Result files will be written OUTSIDE the repo:", OUTPUT_ROOT.resolve())

SEASONAL_TYPES = ("single_seasonality", "multiple_seasonality", "sarma", "sarima")
NONSEASONAL_TYPES = ("ar", "ma", "arma", "ari", "ima", "arima")
SERIES_TYPES = SEASONAL_TYPES + NONSEASONAL_TYPES
EXPECTED_PER_TYPE = 20
EXPECTED_LENGTH = 480
CANDIDATE_PERIODS = [7, 12, 24, 52]  # from params.json: population-level grid, NOT series labels
MIN_CYCLES = 6
MAX_DETECTED_PERIODS = 3

# Original-notebook thresholds: deliberately NOT calibrated on this 80-series evaluation set.
PGRAM_THRESHOLD = 0.08
STL_THRESHOLD = 0.30
FOURIER_BIC_THRESHOLD = 15.0
MIN_PASS_COUNT = 3
TOP_CANDIDATES = 8
DETREND_METHOD = "constant"

RUN_MSTL_ASSISTED_DETECTION = True
SAVE_EXAMPLE_PLOTS = False

statsmodels: 0.14.5
Result files will be written OUTSIDE the repo: D:\1001TS-Stationary\cemre\BeTiSe_period_detection_200series_results


## 2. Load the new Requested Generation data (never the old generator)

Each output folder contains `1-way/part-*.parquet`. Group by `series_id` **within its own folder**, because each Requested Generation run starts IDs over at 1. `seasonality_periods` and `fourier_coefficients` are repeated metadata columns; **never provide them to the detector**.

**Nonseasonal controls:** AR/MA/ARMA/ARI/IMA/ARIMA are loaded from their matching output folders. Missing `seasonality_periods` metadata is acceptable **only** for these six predefined nonseasonal types; it is not accepted for seasonal types. Each nonseasonal series has `true_periods=[]` for evaluation.


In [2]:
def parse_serialized(value):
    if value is None:
        return None
    if isinstance(value, np.ndarray):
        return [parse_serialized(v) for v in value.tolist()]
    if isinstance(value, (list, tuple)):
        return [parse_serialized(v) for v in value]
    if isinstance(value, dict):
        return {k: parse_serialized(v) for k, v in value.items()}
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, str):
        v = value.strip()
        if not v or v.lower() in ("none", "null", "nan"):
            return None
        try:
            return parse_serialized(json.loads(v))
        except (TypeError, ValueError):
            try:
                return parse_serialized(ast.literal_eval(v))
            except (ValueError, SyntaxError):
                return v
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    return value


def parse_periods(value):
    value = parse_serialized(value)
    if value is None:
        return []
    if not isinstance(value, (list, tuple)):
        value = [value]
    periods = []
    for p in value:
        if p is not None and np.isfinite(float(p)):
            periods.append(int(round(float(p))))
    return sorted(set(periods))


def _one_value(df, name):
    return df[name].iloc[0] if name in df.columns else None


def load_requested_series(root=DATA_ROOT, types=SERIES_TYPES):
    """Returns list of independent series records; labels are evaluation-only."""
    records = []
    for series_type in types:
        paths = sorted((Path(root) / series_type / "1-way").glob("part-*.parquet"))
        if not paths:
            raise FileNotFoundError(
                f"No Parquet for {series_type}. Expected: {root}/{series_type}/1-way/part-*.parquet. "
                "First run Requested Generation with num_series=20 for every type."
            )
        try:
            df = pd.concat([pd.read_parquet(p) for p in paths], ignore_index=True)
        except ImportError as exc:
            raise ImportError("Reading Parquet requires pyarrow or fastparquet (pip install pyarrow).") from exc
        needed = {"series_id", "data"}
        if series_type in SEASONAL_TYPES:
            needed.add("seasonality_periods")
        if not needed.issubset(df.columns):
            raise ValueError(f"{series_type} is missing required columns: {sorted(needed-set(df.columns))}")
        for sid, group in df.groupby("series_id", sort=True):
            if "time" in group.columns:
                group = group.sort_values("time")
            x = group["data"].to_numpy(dtype=float)
            if len(x) != EXPECTED_LENGTH or not np.all(np.isfinite(x)):
                raise ValueError(f"Invalid or missing observations in {series_type} series_id={sid}")
            actual_periods = parse_periods(_one_value(group, "seasonality_periods"))
            if series_type in SEASONAL_TYPES:
                if not actual_periods:
                    raise ValueError(f"Missing ground truth periods: {series_type} series_id={sid}")
                true_periods = actual_periods
            elif series_type in NONSEASONAL_TYPES:
                if actual_periods:
                    raise ValueError(f"Nonseasonal control {series_type} {sid} unexpectedly has periods={actual_periods}")
                # Explicit negative-control ground truth, independent of detector input.
                true_periods = []
            else:
                raise ValueError(f"Unrecognized series type: {series_type}")
            features = parse_serialized(_one_value(group, "feature_components"))
            if features not in (None, []):
                raise ValueError(f"Expected base-only series, got features={features} in {series_type} {sid}")
            records.append({
                "key": f"{series_type}__{sid}",
                "series_type": series_type,
                "series_id": sid,
                "x": x,
                "true_periods": true_periods,
                "fourier_coefficients": parse_serialized(_one_value(group, "fourier_coefficients")),
                "seasonality_scale_factor": parse_serialized(_one_value(group, "seasonality_scale_factor")),
                "seasonal_difference": parse_serialized(_one_value(group, "seasonal_difference")),
            })
        count = sum(r["series_type"] == series_type for r in records)
        if count != EXPECTED_PER_TYPE:
            warnings.warn(f"{series_type}: found {count} series; expected {EXPECTED_PER_TYPE}.")
    return records

## 3. Candidate evidence — preserving the original notebook approach

This section retains its four evidence sources: normalized local periodogram power, absolute ACF at the candidate lag, robust STL strength, and linear-trend-adjusted Fourier regression BIC improvement. The initial **fast screening** ranks all candidates using periodogram + ACF; expensive fits operate only on the shortlist.

The cutoffs are experimental defaults from the old notebook, not claims of statistical calibration. In particular, stochastic SARMA need not produce a strong deterministic Fourier signature.

In [3]:
def valid_candidates(n, candidate_periods=CANDIDATE_PERIODS, min_cycles=MIN_CYCLES):
    return sorted({int(p) for p in candidate_periods if 2 <= int(p) <= n // min_cycles})


def centered_series(x):
    x = np.asarray(x, dtype=float)
    if x.ndim != 1 or not len(x) or not np.all(np.isfinite(x)):
        raise ValueError("Expected one complete finite time series")
    return x


def preprocess_for_spectrum(x, method=DETREND_METHOD):
    x = centered_series(x)
    if method == "constant":
        return x - np.mean(x)
    if method == "linear":
        return scipy_detrend(x, type="linear")
    if method is None:
        return x.copy()
    raise ValueError("Unknown detrend method")


def power_share(x, period, window=1):
    x = preprocess_for_spectrum(x)
    n = len(x)
    spectrum = (2.0/n)*np.abs(np.fft.rfft(x))**2
    spectrum[0] = 0.0
    if n % 2 == 0:
        spectrum[-1] = 0.0
    total = spectrum.sum()
    target = int(round(n/period))
    if total <= 0 or not (0 < target < len(spectrum)):
        return 0.0
    lo, hi = max(1, target-window), min(len(spectrum)-1, target+window)
    return float(np.max(spectrum[lo:hi+1])/total)


def autocorrelation_at_lag(x, lag):
    x = centered_series(x)-np.mean(x)
    d = np.sum(x*x)
    return float(np.sum(x[:-lag]*x[lag:])/d) if 0 < lag < len(x) and d>0 else np.nan


def component_strength(resid, seasonal):
    resid = np.asarray(resid)
    seasonal = np.asarray(seasonal)
    den = np.var(resid+seasonal)
    return float(np.clip(1.0-np.var(resid)/den, 0.0, 1.0)) if den>0 else 0.0


def stl_strength(x, p):
    if len(x)<2*p:
        return np.nan
    return component_strength(*(
        lambda f: (f.resid, f.seasonal)
    )(STL(x, period=int(p), robust=True).fit()))


def fourier_matrix(n, periods, harmonics=1):
    t = np.arange(n)
    cols = []
    for p in periods:
        for k in range(1, harmonics+1):
            cols.extend([np.sin(2*np.pi*k*t/p), np.cos(2*np.pi*k*t/p)])
    return np.column_stack(cols) if cols else np.empty((n,0))


def fourier_bic_improvement(x, p, harmonics=1):
    n = len(x)
    t = np.arange(n)
    z = (t-np.mean(t))/(np.std(t)+1e-12)
    baseline = np.column_stack([np.ones(n), z])
    full = np.column_stack([baseline, fourier_matrix(n, [p], harmonics)])
    rss0 = np.sum((x-baseline@np.linalg.lstsq(baseline,x,rcond=None)[0])**2)
    rss1 = np.sum((x-full@np.linalg.lstsq(full,x,rcond=None)[0])**2)
    bic0 = n*np.log(max(rss0,1e-12)/n)+baseline.shape[1]*np.log(n)
    bic1 = n*np.log(max(rss1,1e-12)/n)+full.shape[1]*np.log(n)
    return float(bic0-bic1)

## 4. Comparison within **period detection only**

The **STL baseline** fits each candidate period on its own and adds its seasonal-strength vote. The **MSTL-assisted** method uses the **same** candidate grid, spectral/ACF evidence, Fourier/BIC score and voting threshold, but for the two highest-ranked **preliminary** candidates it fits one joint MSTL model and obtains a separate conditional strength from each estimated seasonal component.

The two candidate periods are selected from data, **not the generator metadata**. In particular, MSTL cannot discover a completely unproposed period. With multiple real periods, the top-two restriction is a limitation we want the evaluation to expose; we should not silently force the true periods into the MSTL call.

For components \(S_t^{(s_j)}\) and the joint MSTL residual \(R_t\), the *experimental component-level* score is

\[
F_{s_j}^{MSTL}=\max\left(0,\ 1-rac{\operatorname{Var}(R_t)}{\operatorname{Var}(R_t+S_t^{(s_j)})}ight).
\]

This is a **conditional diagnostic**, not a guaranteed attribution of true seasonal variation; shared harmonics (e.g., 12/24) and overfitting can complicate interpretation. The existing `STL_THRESHOLD` is provisionally reused so results are comparable, not because it is universally calibrated for both scores.


In [4]:
def rescore(df):
    df = df.copy()
    df["pass_periodogram"] = df["periodogram_power_share"] >= PGRAM_THRESHOLD
    df["pass_acf"] = df["acf_abs"] >= df["acf_threshold"]
    df["pass_decomposition"] = df["decomposition_strength"] >= STL_THRESHOLD
    df["pass_fourier"] = df["bic_improvement"] >= FOURIER_BIC_THRESHOLD
    votes = ["pass_periodogram", "pass_acf", "pass_decomposition", "pass_fourier"]
    df["pass_count"] = df[votes].fillna(False).astype(int).sum(axis=1)
    df["final_score"] = (2*df["periodogram_power_share"] + df["acf_abs"]
        + 1.5*df["decomposition_strength"].fillna(0)
        + 0.05*df["bic_improvement"].fillna(0).clip(lower=0))
    df["accepted"] = df["pass_count"] >= MIN_PASS_COUNT
    return df.sort_values("final_score", ascending=False).reset_index(drop=True)


def detect_periods(x, method="stl"):
    """Returns (predicted integer periods, candidate-score dataframe). No metadata passed."""
    if method not in ("stl", "mstl_top2"):
        raise ValueError("method must be 'stl' or 'mstl_top2'")
    x = centered_series(x)
    candidates = valid_candidates(len(x))
    base = []
    for p in candidates:
        acf = autocorrelation_at_lag(x, p)
        share = power_share(x, p)
        base.append({
            "period":p, "periodogram_power_share":share,
            "acf":acf, "acf_abs":abs(acf) if np.isfinite(acf) else 0.0,
            "acf_threshold":2.0/np.sqrt(len(x)),
            "preliminary_score":2.0*share+(abs(acf) if np.isfinite(acf) else 0.0),
        })
    scores = pd.DataFrame(base)
    if scores.empty:
        return [], scores
    scores = scores.sort_values("preliminary_score", ascending=False).reset_index(drop=True)
    top = scores.head(min(TOP_CANDIDATES,len(scores)))["period"].tolist()
    scores["decomposition_strength"] = np.nan
    scores["bic_improvement"] = np.nan
    scores["mstl_strength_used"] = False
    for p in top:
        idx = scores.index[scores["period"]==p][0]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            scores.loc[idx,"decomposition_strength"] = stl_strength(x,p)
            scores.loc[idx,"bic_improvement"] = fourier_bic_improvement(x,p)
    if method == "mstl_top2" and len(top)>=2:
        # top-2 candidates chosen entirely from DATA (preliminary spectral + ACF ranks)
        chosen = sorted(top[:2])
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            model = MSTL(x, periods=chosen, stl_kwargs={"robust": True}).fit()
        components = np.asarray(model.seasonal)
        if components.ndim == 1:
            components = components.reshape(-1,1)
        for k,p in enumerate(chosen):
            idx = scores.index[scores["period"]==p][0]
            scores.loc[idx,"decomposition_strength"] = component_strength(model.resid, components[:,k])
            scores.loc[idx,"mstl_strength_used"] = True
    scores = rescore(scores)
    periods = scores.loc[scores["accepted"],"period"].head(MAX_DETECTED_PERIODS).astype(int).tolist()
    return sorted(periods), scores

## 5. Exact period-level and per-series evaluation

A result is **exact-set correct** only when predicted periods precisely match the full true set. For multiple seasonality `[12, 24]`, detecting only `[12]` is not fully correct. No tolerance is used. Candidate-level TP/FP/FN/TN are reported separately; **TN is relative to the chosen candidate grid**, not evidence of nonseasonality specificity. We have no negative control series in this initial 80.

For nonseasonal controls the correct prediction is the empty set `[]`. Thus `exact_set_accuracy` on each nonseasonal group is its **no-false-alarm rate (specificity at the series level)**; `1 - exact_set_accuracy` is the proportion of series for which the detector reported at least one false period. The `any_found_rate` metric is not informative on these six negative-control groups because no true period exists.


In [5]:
def evaluate_one(true_periods, predicted_periods, grid):
    truth = set(map(int,true_periods))
    pred = set(map(int,predicted_periods))
    grid = set(map(int,grid))
    if not truth.issubset(grid):
        raise ValueError(f"TRUE periods {sorted(truth-grid)} absent from search grid; cannot evaluate fairly")
    tp, fp, fn, tn = len(truth&pred), len(pred-truth), len(truth-pred), len(grid-(truth|pred))
    return {
        "TP":tp, "FP":fp, "FN":fn, "TN":tn,
        "exact_set_match":truth==pred,
        "any_correct":bool(truth&pred),
        "all_true_found":truth.issubset(pred),
        "has_extra":bool(pred-truth),
    }


def evaluate_records(records, methods=("stl", "mstl_top2")):
    outcomes, score_tables, errors = [], [], []
    for i, rec in enumerate(records, start=1):
        grid = valid_candidates(len(rec["x"]))
        for method in methods:
            try:
                pred, scores = detect_periods(rec["x"], method=method)
                outcome = evaluate_one(rec["true_periods"], pred, grid)
                outcomes.append({
                    "key":rec["key"], "series_type":rec["series_type"],
                    "series_id":rec["series_id"], "n":len(rec["x"]),
                    "method":method,
                    "true_periods":json.dumps(rec["true_periods"]),
                    "predicted_periods":json.dumps(pred),
                    **outcome,
                })
                scores = scores.copy()
                scores.insert(0,"key",rec["key"])
                scores.insert(1,"method",method)
                score_tables.append(scores)
            except Exception as exc:
                errors.append({"key":rec["key"],"method":method,"error":repr(exc)})
        if i % 20 == 0:
            print(f"Evaluated {i}/{len(records)} series")
    return pd.DataFrame(outcomes), pd.concat(score_tables,ignore_index=True) if score_tables else pd.DataFrame(), pd.DataFrame(errors)


def summary_table(results):
    if results.empty:
        return pd.DataFrame()
    s = results.groupby(["series_type","method"],as_index=False).agg(
        n=("key","count"),
        exact_set_accuracy=("exact_set_match","mean"),
        any_found_rate=("any_correct","mean"),
        all_found_rate=("all_true_found","mean"),
        TP=("TP","sum"), FP=("FP","sum"), FN=("FN","sum"), TN=("TN","sum"),
    )
    s["period_precision"] = s["TP"]/(s["TP"]+s["FP"]).replace(0,np.nan)
    s["period_recall"] = s["TP"]/(s["TP"]+s["FN"]).replace(0,np.nan)
    return s

## 6. Execute STL and MSTL on all 200 generated series

You must first generate the original 80 seasonal series and six new nonseasonal sets (20 each). If a folder is missing, the notebook will stop rather than fabricate results. On the 120 controls, an empty prediction is correct; any selected period counts as a false alarm.


In [6]:
records = load_requested_series()
print("Loaded:",pd.Series([r["series_type"] for r in records]).value_counts().to_dict())

methods = ("stl","mstl_top2") if RUN_MSTL_ASSISTED_DETECTION else ("stl",)
results_df, candidate_scores_df, errors_df = evaluate_records(records, methods=methods)
summary_df = summary_table(results_df)

results_df.to_csv(OUTPUT_ROOT/"period_predictions_by_series.csv",index=False)
candidate_scores_df.to_csv(OUTPUT_ROOT/"candidate_scores.csv",index=False)
summary_df.to_csv(OUTPUT_ROOT/"period_accuracy_by_type_and_method.csv",index=False)
errors_df.to_csv(OUTPUT_ROOT/"errors.csv",index=False)

print("Successful method-series runs:",len(results_df),"; Errors:",len(errors_df))
display(summary_df)
print("\n=== Nonseasonal false alarms (fraction of series with at least one detected period) ===")
negative_summary = summary_df[summary_df["series_type"].isin(NONSEASONAL_TYPES)].copy()
negative_summary["false_alarm_rate"] = 1.0 - negative_summary["exact_set_accuracy"]
display(negative_summary[["series_type", "method", "n", "false_alarm_rate", "FP"]])
if not errors_df.empty:
    display(errors_df)

Loaded: {'single_seasonality': 20, 'multiple_seasonality': 20, 'sarma': 20, 'sarima': 20, 'ar': 20, 'ma': 20, 'arma': 20, 'ari': 20, 'ima': 20, 'arima': 20}
Evaluated 20/200 series
Evaluated 40/200 series
Evaluated 60/200 series
Evaluated 80/200 series
Evaluated 100/200 series
Evaluated 120/200 series
Evaluated 140/200 series
Evaluated 160/200 series
Evaluated 180/200 series
Evaluated 200/200 series
Successful method-series runs: 400 ; Errors: 0


,series_type,method,n,exact_set_accuracy,any_found_rate,all_found_rate,TP,FP,FN,TN,period_precision,period_recall
0,ar,mstl_top2,20,1.00,0.00,1.00,0,0,0,80,NaN,NaN
1,ar,stl,20,1.00,0.00,1.00,0,0,0,80,NaN,NaN
2,ari,mstl_top2,20,1.00,0.00,1.00,0,0,0,80,NaN,NaN
3,ari,stl,20,1.00,0.00,1.00,0,0,0,80,NaN,NaN
4,arima,mstl_top2,20,0.95,0.00,1.00,0,1,0,79,0.000000,NaN
5,arima,stl,20,0.95,0.00,1.00,0,1,0,79,0.000000,NaN
6,arma,mstl_top2,20,0.95,0.00,1.00,0,1,0,79,0.000000,NaN
7,arma,stl,20,0.95,0.00,1.00,0,1,0,79,0.000000,NaN
8,ima,mstl_top2,20,1.00,0.00,1.00,0,0,0,80,NaN,NaN
9,ima,stl,20,1.00,0.00,1.00,0,0,0,80,NaN,NaN



=== Nonseasonal false alarms (fraction of series with at least one detected period) ===


,series_type,method,n,false_alarm_rate,FP
0,ar,mstl_top2,20,0.00,0
1,ar,stl,20,0.00,0
2,ari,mstl_top2,20,0.00,0
3,ari,stl,20,0.00,0
4,arima,mstl_top2,20,0.05,1
5,arima,stl,20,0.05,1
6,arma,mstl_top2,20,0.05,1
7,arma,stl,20,0.05,1
8,ima,mstl_top2,20,0.00,0
9,ima,stl,20,0.00,0


## 7. ACF + stationary Seasonal ARMA model comparison: **independent diagnostic experiment**

We preserve STL/MSTL, their thresholds, and their voting unchanged. In parallel, for each numeric series:

1. **ACF preselection:** From `[7, 12, 24, 52]`, rank lags by absolute autocorrelation; test at most the leading lag if its ACF clears a small preliminary cutoff. This lag is only a *candidate*, not automatically an accepted seasonal period.
2. **Nonseasonal reference:** Fit ARMA(1,1) using `SARIMAX(..., order=(1,0,1), seasonal_order=(0,0,0,0))`.
3. **Seasonal alternatives, ALL `D=0`:** At the nominated lag, fit multiplicative seasonal AR(1), seasonal MA(1), and seasonal ARMA(1,1) variants.
4. **Evidence:** Compare candidate vs reference using a common late-window, penalized one-step innovation score (named `bic_common_window`). Require `bic_gain >= 10` to issue a period proposal, otherwise propose `[]`. This is exploratory **in-sample model evidence**, not a validated hypothesis test.

The new evidence cannot by itself distinguish a deterministic seasonality from a stochastic one. In particular, **ARI/IMA/ARIMA have nonseasonal integration**: their low-frequency/nonstationary behavior can confuse an ARMA(1,1) reference, leading to spurious seasonal proposals. That is why the nonseasonal controls are essential.

**Interpretation of the two new result columns:**

| Example | MSTL says | Stochastic module says | `stochastic_proposal_only` | `fallback_only` |
|---|---|---|---|---|
| Seasonal series where MSTL missed it | `[]` | `[12]` | `[12]` | `[12]` |
| Single-seasonal where STL/MSTL succeeded | `[12]` | `[24]` | `[24]` | `[12]` |
| Nonseasonal control (danger!) | `[]` | `[12]` | `[12]` **false alarm** | `[12]` **false alarm** |
| Incorrect nonempty MSTL result | `[7]` | `[24]` | `[24]` | `[7]` (unchanged, still wrong) |

`stochastic_proposal_only` reports what the *new module alone* believes. `fallback_only` is a **what-if rule**, not a deployed combination: trust any nonempty MSTL set and use the new module only when the MSTL set is empty. A single wrong MSTL result cannot be fixed by fallback, and a nonseasonal false positive can be introduced by it.

Model fitting can take a while for 200 series; progress is printed every 5 series. Convergence failures are reported explicitly in a separate CSV and must not be taken as positive seasonality evidence. **Do not tune the threshold on the same 200 series and then claim unbiased generalization.**


In [7]:
# All stochastic-evidence knobs are separate from the original STL/MSTL thresholds.
# First pass: one ACF-leading period per series keeps fitting cost limited.
STOCH_TOP_ACF_CANDIDATES = 1
STOCH_ACF_MULTIPLIER = 1.0     # require |ACF(s)| >= multiplier * 2/sqrt(n)
STOCH_MIN_BIC_GAIN = 10.0     # exploratory threshold, NOT optimized on the 80 series
STOCH_MAXITER = 45
STOCH_BASELINE_ORDER = (1, 0, 1)
# This independent proposal module tests at most one ACF-selected period.
# It is not expected to recover both periods of a multiple-seasonal series.
STOCH_MAX_NEW_PERIODS = 1

from statsmodels.tsa.statespace.sarimax import SARIMAX
import time


def select_acf_stochastic_candidates(x):
    """Purely observation-based preselection (never reads true periods/types)."""
    x = centered_series(x)
    n = len(x)
    cutoff = STOCH_ACF_MULTIPLIER * 2.0 / np.sqrt(n)
    rows = []
    for period in valid_candidates(n):
        acf = autocorrelation_at_lag(x, period)
        if np.isfinite(acf):
            rows.append({"period": int(period), "acf": float(acf),
                         "acf_abs": float(abs(acf)), "acf_cutoff": float(cutoff)})
    rows = sorted(rows, key=lambda r: r["acf_abs"], reverse=True)
    selected = [r for r in rows if r["acf_abs"] >= cutoff]
    return selected[:STOCH_TOP_ACF_CANDIDATES]


def common_window_bic(fit, first_obs):
    """Same observation window and penalty convention for every candidate.

    Approximate Gaussian one-step innovation log score on a shared late window;
    this is not statsmodels' built-in BIC and is exploratory/in-sample.
    """
    e = np.asarray(fit.filter_results.forecasts_error, dtype=float)[0, first_obs:]
    v = np.asarray(fit.filter_results.forecasts_error_cov, dtype=float)[0, 0, first_obs:]
    good = np.isfinite(e) & np.isfinite(v) & (v > 1e-12)
    if not np.all(good) or len(e) < 30:
        return np.nan
    log_score = -0.5 * np.sum(np.log(2.0 * np.pi * v) + (e * e) / v)
    k = len(fit.params)
    return float(-2.0 * log_score + k * np.log(len(e)))


def fit_seasonal_comparison_model(x, seasonal_order, shared_start):
    """Actual SARIMAX ARMA vs stationary multiplicative seasonal AR/MA fitting."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            mod = SARIMAX(
                x, order=STOCH_BASELINE_ORDER,
                seasonal_order=seasonal_order,
                trend="n", simple_differencing=False,
                enforce_stationarity=False, enforce_invertibility=False,
            )
            fit = mod.fit(disp=False, maxiter=STOCH_MAXITER)
            converged = bool(fit.mle_retvals.get("converged", True))
            score = common_window_bic(fit, shared_start) if converged else np.nan
            return {"bic_common_window": score, "converged": converged,
                    "error": "" if converged else "optimizer_not_converged"}
        except Exception as exc:
            return {"bic_common_window": np.nan, "converged": False,
                    "error": f"{type(exc).__name__}: {str(exc)[:220]}"}


def stochastic_model_evidence(x):
    """(proposed_periods, one-row-per-fit diagnostics), *without any labels*."""
    x = centered_series(x)
    selected = select_acf_stochastic_candidates(x)
    # Include every candidate period in the same fixed common scoring window.
    shared_start = max(2 * max(valid_candidates(len(x))) + 10, 20)
    if len(x) - shared_start < 30:
        raise ValueError("Insufficient common-window observations for seasonal model comparison")

    baseline = fit_seasonal_comparison_model(x, (0, 0, 0, 0), shared_start)
    base_bic = baseline["bic_common_window"]
    fits = [{"period": np.nan, "model": "ARMA_null", "acf": np.nan,
             "acf_abs": np.nan, "acf_cutoff": np.nan, "bic_gain": np.nan,
             **baseline}]
    if not np.isfinite(base_bic):
        return [], pd.DataFrame(fits)

    for item in selected:
        p = int(item["period"])
        alternatives = [
            ("seasonal_AR_D0", (1, 0, 0, p)),
            ("seasonal_MA_D0", (0, 0, 1, p)),
            ("seasonal_ARMA_D0", (1, 0, 1, p)),
        ]
        for model_name, so in alternatives:
            result = fit_seasonal_comparison_model(x, so, shared_start)
            score = result["bic_common_window"]
            fits.append({"period": p, "model": model_name,
                         "acf": item["acf"], "acf_abs": item["acf_abs"],
                         "acf_cutoff": item["acf_cutoff"],
                         "bic_gain": float(base_bic-score) if np.isfinite(score) else np.nan,
                         **result})

    table = pd.DataFrame(fits)
    eligible = table[(table["model"] != "ARMA_null") &
                     (table["bic_gain"] >= STOCH_MIN_BIC_GAIN)]
    if eligible.empty:
        return [], table
    # A period can be supported by multiple seasonal alternatives: keep its best.
    best_periods = (eligible.sort_values("bic_gain", ascending=False)
                    .drop_duplicates("period").head(STOCH_MAX_NEW_PERIODS))
    return sorted(best_periods["period"].astype(int).tolist()), table


In [8]:
# Shadow comparison — keep the STL and MSTL baselines unchanged.
# Requires the original run cell above to have populated `records` and `results_df`.

def run_shadow_stochastic_evaluation(records, old_results):
    baseline = old_results[old_results["method"] == "mstl_top2"]
    old_by_key = {row["key"]: parse_periods(row["predicted_periods"])
                  for _, row in baseline.iterrows()}
    if len(old_by_key) != len(records):
        raise ValueError("Missing MSTL baseline outputs for some series; run original evaluation first")

    fits_out, per_series, outcomes = [], [], []
    t0 = time.monotonic()
    for i, rec in enumerate(records, 1):
        proposal, fit_scores = stochastic_model_evidence(rec["x"])
        fit_scores = fit_scores.copy()
        fit_scores.insert(0, "key", rec["key"])
        fit_scores.insert(1, "series_type_for_evaluation", rec["series_type"])
        fits_out.append(fit_scores)
        old = old_by_key[rec["key"]]
        # Diagnostic fallback: NEVER overwrite a nonempty MSTL prediction.
        fallback = old if old else proposal
        fallback = sorted(set(map(int, fallback)))
        per_series.append({
            "key":rec["key"], "series_type":rec["series_type"],
            "series_id":rec["series_id"],
            "true_periods_evaluation_only":json.dumps(rec["true_periods"]),
            "baseline_mstl":json.dumps(old),
            "stochastic_proposal":json.dumps(proposal),
            "fallback_only":json.dumps(fallback),
            "baseline_changed_by_fallback":(old != fallback),
        })
        grid = valid_candidates(len(rec["x"]))
        for method, pred in [("stochastic_proposal_only", proposal),
                             ("fallback_only", fallback)]:
            outcomes.append({
                "key":rec["key"], "series_type":rec["series_type"],
                "series_id":rec["series_id"], "n":len(rec["x"]),
                "method":method,
                "true_periods":json.dumps(rec["true_periods"]),
                "predicted_periods":json.dumps(pred),
                **evaluate_one(rec["true_periods"], pred, grid),
            })
        if i % 5 == 0 or i == len(records):
            print(f"Stochastic models: {i}/{len(records)} series ({time.monotonic()-t0:.0f}s)")

    fit_df = pd.concat(fits_out, ignore_index=True)
    series_df = pd.DataFrame(per_series)
    shadow_results = pd.DataFrame(outcomes)
    combined = pd.concat([old_results, shadow_results], ignore_index=True)
    return fit_df, series_df, combined


stochastic_fits_df, stochastic_proposals_df, comparison_results_df = (
    run_shadow_stochastic_evaluation(records, results_df)
)
comparison_summary_df = summary_table(comparison_results_df)

# New names deliberately do NOT overwrite the previous STL/MSTL result files.
stochastic_fits_df.to_csv(OUTPUT_ROOT / "stochastic_model_comparison.csv", index=False)
stochastic_proposals_df.to_csv(OUTPUT_ROOT / "stochastic_proposals_by_series.csv", index=False)
comparison_summary_df.to_csv(OUTPUT_ROOT / "stochastic_comparison_accuracy.csv", index=False)
failed_fits_df = stochastic_fits_df[~stochastic_fits_df["converged"].astype(bool)]
failed_fits_df.to_csv(OUTPUT_ROOT / "stochastic_model_fit_failures.csv", index=False)

print("\n=== Exact-set accuracy: unchanged baselines vs new proposals ===")
display(comparison_summary_df[["series_type", "method", "n", "exact_set_accuracy", "TP", "FP", "FN"]]
        .sort_values(["series_type", "method"]))
print("\n=== Baseline changed by fallback, by type ===")
display(stochastic_proposals_df.groupby("series_type", as_index=False)
        .agg(n=("key","count"), changed=("baseline_changed_by_fallback","sum")))
print("\n=== Nonseasonal false alarms, including stochastic-only and fallback ===")
negative_comparison = comparison_summary_df[comparison_summary_df["series_type"].isin(NONSEASONAL_TYPES)].copy()
negative_comparison["false_alarm_rate"] = 1.0 - negative_comparison["exact_set_accuracy"]
display(negative_comparison[["series_type", "method", "n", "false_alarm_rate", "FP"]]
        .sort_values(["series_type", "method"]))
negative_comparison.to_csv(OUTPUT_ROOT / "nonseasonal_false_alarm_rates.csv", index=False)
print("\nNon-converged/failed fits:", len(failed_fits_df), "of", len(stochastic_fits_df), "fits")
print("Result files:", OUTPUT_ROOT.resolve())


Stochastic models: 5/200 series (4s)
Stochastic models: 10/200 series (8s)
Stochastic models: 15/200 series (13s)
Stochastic models: 20/200 series (17s)
Stochastic models: 25/200 series (29s)
Stochastic models: 30/200 series (42s)
Stochastic models: 35/200 series (50s)
Stochastic models: 40/200 series (59s)
Stochastic models: 45/200 series (72s)
Stochastic models: 50/200 series (76s)
Stochastic models: 55/200 series (89s)
Stochastic models: 60/200 series (95s)
Stochastic models: 65/200 series (111s)
Stochastic models: 70/200 series (121s)
Stochastic models: 75/200 series (125s)
Stochastic models: 80/200 series (129s)
Stochastic models: 85/200 series (130s)
Stochastic models: 90/200 series (136s)
Stochastic models: 95/200 series (138s)
Stochastic models: 100/200 series (140s)
Stochastic models: 105/200 series (144s)
Stochastic models: 110/200 series (145s)
Stochastic models: 115/200 series (145s)
Stochastic models: 120/200 series (159s)
Stochastic models: 125/200 series (160s)
Stochasti

,series_type,method,n,exact_set_accuracy,TP,FP,FN
0,ar,fallback_only,20,0.70,0,6,0
1,ar,mstl_top2,20,1.00,0,0,0
2,ar,stl,20,1.00,0,0,0
3,ar,stochastic_proposal_only,20,0.70,0,6,0
4,ari,fallback_only,20,0.90,0,2,0
5,ari,mstl_top2,20,1.00,0,0,0
6,ari,stl,20,1.00,0,0,0
7,ari,stochastic_proposal_only,20,0.90,0,2,0
8,arima,fallback_only,20,0.70,0,6,0
9,arima,mstl_top2,20,0.95,0,1,0



=== Baseline changed by fallback, by type ===


,series_type,n,changed
0,ar,20,6
1,ari,20,2
2,arima,20,5
3,arma,20,3
4,ima,20,0
5,ma,20,0
6,multiple_seasonality,20,0
7,sarima,20,5
8,sarma,20,16
9,single_seasonality,20,0



=== Nonseasonal false alarms, including stochastic-only and fallback ===


,series_type,method,n,false_alarm_rate,FP
0,ar,fallback_only,20,0.30,6
1,ar,mstl_top2,20,0.00,0
2,ar,stl,20,0.00,0
3,ar,stochastic_proposal_only,20,0.30,6
4,ari,fallback_only,20,0.10,2
5,ari,mstl_top2,20,0.00,0
6,ari,stl,20,0.00,0
7,ari,stochastic_proposal_only,20,0.10,2
8,arima,fallback_only,20,0.30,6
9,arima,mstl_top2,20,0.05,1



Non-converged/failed fits: 37 of 704 fits
Result files: D:\1001TS-Stationary\cemre\BeTiSe_period_detection_200series_results


### How to read the output

- **`stl` / `mstl_top2`**: Your existing baseline predictions, **never modified by the new model**.
- **`stochastic_proposal_only`**: Predict using *only* the ACF-leading candidate and stationary seasonal AR/MA-vs-ARMA model score. Zero or one period; no help from STL/MSTL.
- **`fallback_only`**: If MSTL predicts **any** period, keep its result, even if wrong. If MSTL predicts nothing, take the new proposal. This is a diagnostic thought experiment, **not** the chosen final pipeline.
- **Nonseasonal AR / MA / ARMA / ARI / IMA / ARIMA**: The expected answer is always `[]`. `nonseasonal_false_alarm_rates.csv` shows how often each method incorrectly predicts seasonality.
- **`bic_gain`**: A positive value indicates the low-order seasonal model's common-window penalized innovation score improves on the nonseasonal ARMA(1,1) reference. It is **not** proof of stochastic seasonality.
- **Fit errors and convergence**: Inspect `stochastic_model_fit_failures.csv`; failed fits are ineligible for recommending a period.
- **Leakage check**: `true_periods` is passed **only** to evaluation, never to candidate generation or model fitting.


## 8. Error analysis: which periods were missed or spuriously added?

The original notebook's aggressive harmonic pruning was especially risky for `[12, 24]`, so exact-set failures and candidate scores are retained for manual inspection. No change to thresholds is automatically made after seeing these results.

In [9]:
bad = results_df.loc[~results_df["exact_set_match"],
    ["key","series_type","method","true_periods","predicted_periods","TP","FP","FN"]]
display(bad.sort_values(["series_type","method","key"]).head(50))

if not results_df.empty:
    print("Exact-set accuracy by model:")
    display(results_df.pivot_table(index="series_type", columns="method",
        values="exact_set_match", aggfunc="mean"))

,key,series_type,method,true_periods,predicted_periods,TP,FP,FN
393,arima__17,arima,mstl_top2,[],[7],0,1,0
392,arima__17,arima,stl,[],[7],0,1,0
251,arma__6,arma,mstl_top2,[],[7],0,1,0
250,arma__6,arma,stl,[],[7],0,1,0
70,multiple_seasonality__16,multiple_seasonality,stl,"[24, 52]",[52],1,0,1
46,multiple_seasonality__4,multiple_seasonality,stl,"[7, 12]",[12],1,0,1
145,sarima__13,sarima,mstl_top2,[24],[],0,0,1
147,sarima__14,sarima,mstl_top2,[7],[],0,0,1
149,sarima__15,sarima,mstl_top2,[24],"[12, 24]",1,1,0
151,sarima__16,sarima,mstl_top2,[7],[],0,0,1


Exact-set accuracy by model:


method,mstl_top2,stl
series_type,,
ar,1.00,1.00
ari,1.00,1.00
arima,0.95,0.95
arma,0.95,0.95
ima,1.00,1.00
ma,1.00,1.00
multiple_seasonality,1.00,0.90
sarima,0.55,0.60
sarma,0.00,0.00


## 9. Optional diagnostic plots for a small number of series

Do not render all 200 series; select a few challenging examples. This does not affect the scores.

In [10]:
def plot_one_case(record, results=results_df):
    import matplotlib.pyplot as plt
    from scipy.signal import periodogram
    x=record["x"]
    fig,axes=plt.subplots(2,1,figsize=(12,6))
    axes[0].plot(x,linewidth=1)
    axes[0].set_title(f"{record['key']} | ground truth: {record['true_periods']}")
    for _,row in results.loc[results["key"]==record["key"]].iterrows():
        axes[0].plot([],[],label=f"{row['method']}: {row['predicted_periods']}")
    axes[0].legend()
    freq,power=periodogram(preprocess_for_spectrum(x))
    ok=freq>0
    periods=1/freq[ok]
    valid=periods<=len(x)//MIN_CYCLES
    axes[1].plot(periods[valid],power[ok][valid])
    for p in record["true_periods"]:
        axes[1].axvline(p,linestyle="--",label=f"true {p}")
    axes[1].set_xlabel("Period (samples)")
    axes[1].set_ylabel("Periodogram power")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

# Example: plot_one_case(records[0])

## 10. Interpretation and next steps

1. **Preserve deterministic accuracy:** Check `single_seasonality` and `multiple_seasonality` exact-set scores. Do not automatically replace an accurate STL/MSTL prediction.
2. **Measure stochastic seasonal recall:** Report SARMA and SARIMA scores **separately**; note that the stochastic comparison currently tests only stationary seasonal AR/MA models (all `D=0`).
3. **Measure nonseasonal false-alarm rate:** For AR/MA/ARMA/ARI/IMA/ARIMA, the ideal predicted period set is `[]`. Interpret `exact_set_accuracy` as no-false-alarm rate for these types; `nonseasonal_false_alarm_rates.csv` is provided for convenience.
4. **No automatic deployment:** The `fallback_only` rule is a diagnostic and can cause new false alarms on negative controls even if it preserves the initial deterministic pilot scores. Decide on a combined mechanism only after reviewing false alarms.
5. **Methodological limits:** The period grid `[7,12,24,52]` is a limited global candidate set; integrated nonseasonal series challenge the ARMA(1,1) reference; multiple seasonality cannot be recovered by a one-period stochastic proposal; these 200 examples are a development set rather than a final held-out benchmark.
6. **Out of scope:** Seasonal unit root testing, deterministic/stochastic classification, deseasonalization, and reconstruction of seasonal components.

This is a standalone notebook. The generated CSVs are saved **outside the repository** in `BeTiSe_period_detection_200series_results/`. Nothing is pushed or written to the BeTiSe GitHub repository.
